# 03 - Stage 6.3: ROI classifier evaluation

Runs the trained 3D DenseNet classifier on held-out val/calibration patients, both with oracle ground-truth ROIs and the stage 6.2 predicted masks. The inference command uses CUDA when available.

In [ ]:
import pandas as pd
import subprocess
import sys
from pathlib import Path
from IPython.display import display
from src.metrics import auroc_by_size_bucket

root = Path.cwd()
split = root / "results" / "review1_local" / "full281" / "split_281cases.json"
checkpoint = root / "checkpoints" / "stage63" / "best.pt"
masks = root / "results" / "final" / "masks"
for name, extra in (("oracle", []), ("real", ["--pred-masks", str(masks)])):
    csv_path = root / "results" / "final" / name / "cls_predictions.csv"
    if not csv_path.exists():
        command = [
            sys.executable, "-m", "src.predict_cls", "--split", str(split),
            "--ckpt", str(checkpoint), "--mc", "20", "--groups", "val", "calib",
            "--out", str(csv_path.parent), *extra,
        ]
        subprocess.run(command, check=True, cwd=root)
    predictions = pd.read_csv(csv_path)
    print(name)
    for group, frame in predictions.groupby("group"):
        print(group, "patches:", len(frame), "unique windows:", frame["case"].nunique())
        display(auroc_by_size_bucket(frame))
        display(frame.head(8))